
# FraudShield AI
## Model Improvement and Final Evaluation

Objectives:
- Create chronological training, validation, and test partitions.
- Train LightGBM using the training partition.
- Select the model iteration using validation data.
- Evaluate final performance once on the untouched test partition.
- Compare precision, recall, F1-score, PR-AUC, and ROC-AUC.


In [1]:

from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import joblib

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

PROJECT_ROOT = Path(r"D:\FraudShield")
DATA_FILE = PROJECT_ROOT / "data" / "processed" / "train_merged.parquet"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(DATA_FILE)
df = df.sort_values("TransactionDT").reset_index(drop=True)

print("Dataset shape:", df.shape)
print("Fraud rate:", round(df["isFraud"].mean() * 100, 4), "%")


Dataset shape: (590540, 434)
Fraud rate: 3.499 %


In [2]:

TARGET = "isFraud"
EXCLUDED = ["isFraud", "TransactionID"]

n = len(df)
train_end = int(n * 0.70)
valid_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
valid_df = df.iloc[train_end:valid_end].copy()
test_df = df.iloc[valid_end:].copy()

feature_columns = [
    col for col in df.columns
    if col not in EXCLUDED
]

X_train = train_df[feature_columns].copy()
X_valid = valid_df[feature_columns].copy()
X_test = test_df[feature_columns].copy()

y_train = train_df[TARGET].astype("int8")
y_valid = valid_df[TARGET].astype("int8")
y_test = test_df[TARGET].astype("int8")

print("Training rows:", len(train_df))
print("Validation rows:", len(valid_df))
print("Test rows:", len(test_df))

print(f"Train fraud rate: {y_train.mean() * 100:.4f}%")
print(f"Validation fraud rate: {y_valid.mean() * 100:.4f}%")
print(f"Test fraud rate: {y_test.mean() * 100:.4f}%")


Training rows: 413378
Validation rows: 88581
Test rows: 88581
Train fraud rate: 3.5169%
Validation fraud rate: 3.4341%
Test fraud rate: 3.4804%


In [3]:

categorical_columns = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

for col in categorical_columns:
    train_values = X_train[col].astype("string").fillna("__MISSING__")
    valid_values = X_valid[col].astype("string").fillna("__MISSING__")
    test_values = X_test[col].astype("string").fillna("__MISSING__")

    # Learn category vocabulary from training data only.
    categories = pd.Index(train_values.unique())

    X_train[col] = pd.Categorical(
        train_values, categories=categories
    )
    X_valid[col] = pd.Categorical(
        valid_values, categories=categories
    )
    X_test[col] = pd.Categorical(
        test_values, categories=categories
    )

positive_count = int(y_train.sum())
negative_count = len(y_train) - positive_count
scale_pos_weight = negative_count / max(positive_count, 1)

print("Categorical features:", len(categorical_columns))
print(f"Positive-class weight: {scale_pos_weight:.2f}")


Categorical features: 31
Positive-class weight: 27.43


In [7]:

model = lgb.LGBMClassifier(
    objective="binary",
    metric="None",
    n_estimators=4000,
    learning_rate=0.05,
    num_leaves=31,
    colsample_bytree=0.8,
    subsample=0.8,
    subsample_freq=1,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=2,
    verbosity=-1,
)

model.fit(
    X_train,
    y_train,
    categorical_feature=categorical_columns,
    eval_set=[(X_valid, y_valid)],
    eval_metric="average_precision",
    callbacks=[
        lgb.early_stopping(100, first_metric_only=True),
        lgb.log_evaluation(100),
    ],
)

print("Best iteration:", model.best_iteration_)


d:\FraudShield\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[100]	valid_0's average_precision: 0.482856
[200]	valid_0's average_precision: 0.516237
[300]	valid_0's average_precision: 0.536568
[400]	valid_0's average_precision: 0.547145
[500]	valid_0's average_precision: 0.557874
[600]	valid_0's average_precision: 0.565454
[700]	valid_0's average_precision: 0.570588
[800]	valid_0's average_precision: 0.575248
[900]	valid_0's average_precision: 0.579325
[1000]	valid_0's average_precision: 0.582769
[1100]	valid_0's average_precision: 0.586018
[1200]	valid_0's average_precision: 0.59066
[1300]	valid_0's average_precision: 0.592925
[1400]	valid_0's average_precision: 0.594968
[1500]	valid_0's average_precision: 0.597812
[1600]	valid_0's average_precision: 0.600008
[1700]	valid_0's average_precision: 0.602314
[1800]	valid_0's average_precision: 0.602892
[1900]	valid_0's average_precision: 0.60468
[2000]	valid_0's average_precision: 0.606009
[2100]	valid_0's average_precision: 0.609321
[220

In [8]:

test_probabilities = model.predict_proba(
    X_test,
    num_iteration=model.best_iteration_,
)[:, 1]

test_threshold = 0.50
test_predictions = (
    test_probabilities >= test_threshold
).astype("int8")

test_metrics = {
    "best_iteration": int(model.best_iteration_ or 0),
    "test_rows": int(len(y_test)),
    "test_fraud_rate_percent": float(y_test.mean() * 100),
    "average_precision": float(
        average_precision_score(y_test, test_probabilities)
    ),
    "roc_auc": float(roc_auc_score(y_test, test_probabilities)),
    "precision_at_050": float(
        precision_score(y_test, test_predictions, zero_division=0)
    ),
    "recall_at_050": float(
        recall_score(y_test, test_predictions, zero_division=0)
    ),
    "f1_at_050": float(
        f1_score(y_test, test_predictions, zero_division=0)
    ),
}

display(pd.Series(test_metrics, name="value").to_frame())

print("\nConfusion matrix:")
print(confusion_matrix(y_test, test_predictions, labels=[0, 1]))

print("\nClassification report:")
print(classification_report(
    y_test,
    test_predictions,
    target_names=["Legitimate", "Fraud"],
    zero_division=0,
))


,value
best_iteration,3118.000000
test_rows,88581.000000
test_fraud_rate_percent,3.480430
average_precision,0.525789
roc_auc,0.889376
precision_at_050,0.730402
recall_at_050,0.371716
f1_at_050,0.492691



Confusion matrix:
[[85075   423]
 [ 1937  1146]]

Classification report:
              precision    recall  f1-score   support

  Legitimate       0.98      1.00      0.99     85498
       Fraud       0.73      0.37      0.49      3083

    accuracy                           0.97     88581
   macro avg       0.85      0.68      0.74     88581
weighted avg       0.97      0.97      0.97     88581



In [9]:

joblib.dump(model, MODEL_DIR / "lightgbm_final_baseline.joblib")

with open(REPORT_DIR / "final_test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_metrics, f, indent=2)

with open(MODEL_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(feature_columns, f, indent=2)

print("Model and test metrics saved successfully.")


Model and test metrics saved successfully.
